# Práctico Clase 1

NOMBRE ALUMNO: **ESCRIBIR NOMBRE AQUI**

Diplomado en Machine Learning Aplicado UC

**Profesor:** Vicente Dominguez

En este práctico compararemos los resultados de algoritmos de recomendación:
- Most popular.
- User-based KNN.
- Item-based KNN.

Utilizaremos la librería **surprise** (https://surpriselib.com/)


## Configuración inicial

In [1]:
# descarga de datasets de train, test e información de items
!gdown 1gmOrtPpZpHJ0HeBwtne-kA8Bll4rFWW7
!gdown 1bnLJUEIRx13k4nxN7x7Fa-3L37rXre73
!gdown 1i92TtKsgf_3ffef8EVLH9NNArxvF-cMo

Downloading...
From: https://drive.google.com/uc?id=1gmOrtPpZpHJ0HeBwtne-kA8Bll4rFWW7
To: /content/u.item
100% 236k/236k [00:00<00:00, 86.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bnLJUEIRx13k4nxN7x7Fa-3L37rXre73
To: /content/u2.base
100% 1.58M/1.58M [00:00<00:00, 47.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1i92TtKsgf_3ffef8EVLH9NNArxvF-cMo
To: /content/u2.test
100% 395k/395k [00:00<00:00, 97.7MB/s]


vemos los nombres de los archivos descargados:

In [2]:
ls

sample_data/  u2.base  u2.test  u.item


instalacion e importacion de librerias:

In [1]:
# instalacion de libreria surprise
!pip uninstall -y numpy scikit-surprise
!pip install numpy==1.24.4
!pip3 install scikit-surprise

Found existing installation: numpy 1.24.4
Uninstalling numpy-1.24.4:
  Successfully uninstalled numpy-1.24.4
Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4
  Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.24.4 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.24.4 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.24.4 which is incompatible.
pymc 5.23.0 requires numpy>=1.25.0, but you have numpy 1.24.4 which is incompatible.
jaxlib 0.5.1 requ

  Using cached scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl


In [2]:
import pandas as pd
from surprise import Reader
from surprise import Dataset
from surprise import NormalPredictor # random rating prediction
from surprise import KNNWithMeans # user-based KNN y item-based KNN
from surprise.accuracy import rmse
import seaborn as sns
from surprise.model_selection import cross_validate

%matplotlib inline
sns.set(style="whitegrid")

## Análisis exploratorio de datos

### Datos de entrenamiento:

In [3]:
df_train = pd.read_csv('u2.base',
                         sep='\t',
                         names=['userid', 'itemid', 'rating', 'timestamp'],
                         header=None)
df_train.head()

,userid,itemid,rating,timestamp
0,1,3,4,878542960
1,1,4,3,876893119
2,1,5,3,889751712
3,1,6,5,887431973
4,1,7,4,875071561


### Datos de test:
- Se tienen que repetir los usuarios y los items del set de entrenamiento.
- La tarea es precedir el rating de estos items.

In [4]:
df_test = pd.read_csv('u2.test',
                         sep='\t',
                         names=['userid', 'itemid', 'rating', 'timestamp'],
                         header=None)
df_test.head()

,userid,itemid,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,8,1,875072484
3,1,9,5,878543541
4,1,21,1,878542772


Nombres de películas y metadata:

In [5]:
# Cargamos el dataset con los items
columns = ['movieid', 'title', 'release_date', 'video_release_date', \
           'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', \
           'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', \
           'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', \
           'Thriller', 'War', 'Western']

df_items = pd.read_csv('u.item',
                        sep='|',
                        index_col=0,
                        names = columns,
                        header=None,
                        encoding='latin-1')

# reset index
df_items = df_items.reset_index()

# diccionario que entrega nombre de la pelicula con el id
dict_item_title = {idx:title for idx, title in zip(df_items.movieid, df_items.title)}

df_items.head()

,movieid,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


Estadísticas de ratings:

In [6]:
df_train.describe()[['rating']]

,rating
count,80000.000000
mean,3.526463
std,1.124429
min,1.000000
25%,3.000000
50%,4.000000
75%,4.000000
max,5.000000


1. **COMENTE LOS RESULTADOS (1 PTO):**
RESPONDE AQUI.

Cuantos usuarios e items distintos tenemos en el set de entrenamiento:

In [7]:
print("Users:", df_train["userid"].nunique())
print("Items:", df_train["itemid"].nunique())

Users: 943
Items: 1648


Número de interacciones por item:

In [8]:
items = df_train.groupby(["itemid"]).count()['userid']
items.describe()

,userid
count,1648.000000
mean,48.543689
std,64.950707
min,1.000000
25%,6.000000
50%,22.000000
75%,65.000000
max,461.000000


2. **COMENTE LOS RESULTADOS (1 PTO):** RESPONDER AQUI

Item con más interacciones (más populares):

In [9]:
most_active_items = items.sort_values(ascending=False)

most_active_items = most_active_items.to_frame().reset_index().rename(columns={"userid": "interactions"})

most_active_items['title'] = [dict_item_title[x] for x in most_active_items.itemid]

most_active_items.head(10)

,itemid,interactions,title
0,50,461,Star Wars (1977)
1,100,414,Fargo (1996)
2,258,409,Contact (1997)
3,181,406,Return of the Jedi (1983)
4,294,402,Liar Liar (1997)
5,286,397,"English Patient, The (1996)"
6,288,360,Scream (1996)
7,1,358,Toy Story (1995)
8,121,351,Independence Day (ID4) (1996)
9,300,345,Air Force One (1997)


3. **COMENTE LOS RESULTADOS (1 PTO):**
RESPONDER AQUI

Concentración de interacciones en 20% de usuarios más activos:

In [10]:
proportion = 0.2
N = int(df_train.userid.nunique() * proportion)
interactions_by_active_items = most_active_items[:N]['interactions'].sum() / items.sum() * 100
print("{:.2f}% de las interacciones viene de los {} usuarios más activos ({:.0f}%)".format(
    interactions_by_active_items, N, proportion*100))

46.16% de las interacciones viene de los 188 usuarios más activos (20%)


4. **COMENTE LOS RESULTADOS (1 PTO):**
RESPONDER AQUI


## Convertir dataframe de Pandas a formato surprise

In [11]:
reader = Reader(rating_scale=(1, 5))
data_train = Dataset.load_from_df(df_train[['userid', 'itemid', 'rating']], reader)
data_test = Dataset.load_from_df(df_test[['userid', 'itemid', 'rating']], reader)

# procesar data para libreria surprise
data_train = data_train.build_full_trainset()
data_test = [data_test.df.loc[i].to_list() for i in range(len(data_test.df))]


## Rating Aleatorio
- En surprise: `NormalPredictor`

In [12]:
algo_rndm = NormalPredictor()
algo_rndm.fit(data_train)
predictions = algo_rndm.test(data_test)
RMSE = rmse(predictions)

RMSE: 1.5235


## Prediccion de rating utilizando informacion de Usuarios más cercanos (KNN-user-based):


In [13]:
sim_options = { 'name': 'cosine' ,'user_based':  True}

algo_knn_user = KNNWithMeans(k = 10 , sim_options = sim_options) # modificar cantidad de vecinos parametro K

algo_knn_user.fit(data_train)
predictions = algo_knn_user.test(data_test)
RMSE = rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9889


## Prediccion de rating utilizando informacion de Items más cercanos (KNN-item-based):


In [14]:
sim_options = { 'name': 'cosine' ,'user_based':  False} # item-based CF

algo_knn_item = KNNWithMeans(k = 10 , sim_options = sim_options) # modificar cantidad de vecinos parametro K

algo_knn_item.fit(data_train)
predictions = algo_knn_item.test(data_test)
RMSE = rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9839


## 5. COMENTE LOS RESULTADOS DE TODOS LOS RECOMENDADORES.

Considere que métrica de error RMSE, donde mientras menores son más acertadas son las recomendaciones.

- ¿Cuál es el algoritmo que obtiene mejor desempeño en terminos de RMSE?

**RESPONDER AQUI (3 PTOS):**

## 6. Análisis de sensibilidad de User-Based KNN (5 PTOS)
- Modificar cantidad de USUARIOS vecinos (K) a 5, 10, 15, 20 , 30 , 50, 60, 70,80, 90 y 100 e imprimir el valor de RMSE para cada uno.


In [ ]:
##### ESCRIBIR CODIGO AQUI ##############################



**7.COMENTE LOS RESULTADOS AQUI SOBRE LA CANTIDAD OPTIMA DE VECINOS (USUARIOS) CERCANOS INDIQUE INTUICIÓN DE POR QUE PASADO CIERTA CANTIDAD DE USUARIOS VECINOS EL RENDIMIENTO COMIENZA A DECRECER?** (5 PTOS)

RESPONDER AQUI.